# Pair-score Experiment 3 — Stage 2: frac-head BC on top of trained pair stack

Loads the **pair-trained checkpoint** from a previous Exp 3 run (`exp3_cross_entity_planet_fleet_Ebi_20260511-125438/pair_score_best.pt`, val_top1 = 0.450 at epoch 25 with all 4 encoders unfrozen) and trains ONLY the new **FracHead** on top — pair head + all encoders are frozen. The combined ckpt that comes out has pair_score_head + frac_head + every encoder state, ready for the downstream PPO bring-up.

**Why stage-2 instead of joint training:** the pair stack's val_top1 = 0.450 is the strongest signal we have so far. Adding the frac loss with the pair head still trainable would slosh the pair representation around and risk regressing it. Stage-2 freezes the pair stack bit-exact and only spends compute on the new ~33k-param frac head.

**What gets trained:** `FracHead` only (~33k params + 1 scalar `frac_log_std`).

**Training runs in-kernel** (`train_pair_score_kwargs`), so every epoch's `tr_frac_mae=… val_frac_mae=… vs_baseline=… frac_sigma=…` line streams into this cell output as it happens.

**Prerequisites in `gs://orbit-wars-shipping/`**: `code.tgz`, `data.tgz`, `weights.tgz`, `pair_score_assets.tgz` from the current repo. Build & upload locally with:

```bash
PAIR_SCORE_PLAYER=Ebi INCLUDE_PAIR_SCORE_ASSETS=1 UPLOAD=1 ./scripts/pack_for_gpu.sh
```

**Runtime**: Runtime → Change runtime type → T4 GPU. Walltime ~5–10 min (frac head is small and only ~33k params train).

## 1. Verify GPU

In [ ]:
import torch, sys
if not torch.cuda.is_available():
    sys.exit('No GPU runtime — Runtime → Change runtime type → T4 GPU, then re-run.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

## 2. Configuration

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT = 'analog-receiver-489214-e9'
BUCKET  = 'gs://orbit-wars-shipping'
PLAYER  = 'Ebi'

# Pair-trained ckpt to resume from. Default: the full-unfreeze
# pair-only run (val_top1 = 0.450, all 4 encoders fine-tuned).
PRIOR_RUN = 'exp3_cross_entity_planet_fleet_Ebi_20260511-125438'
PRIOR_BEST_GCS = f'{BUCKET}/runs/{PRIOR_RUN}/pair_score_best.pt'

# Stage-2 frac-head training: pair head + all encoders frozen,
# only FracHead trains. ``unfreeze=''`` keeps all encoders frozen;
# ``freeze_pair_head=True`` freezes the pair head we just trained.
FRAC_WEIGHT       = 1.0    # loss = frac NLL only (pair head frozen)
FREEZE_PAIR_HEAD  = True
UNFREEZE          = ''     # no encoder thawing in stage-2
LR                = 1e-3   # high-ish; only a small head trains
EPOCHS            = 25
BATCH_SIZE        = 64
VAL_FRAC          = 0.2
MAX_ROWS          = None   # use every acted Ebi snapshot

!gcloud config set project {PROJECT}

## 3. Pull tarballs + prior best.pt

In [ ]:
import os
WORK = '/content/orbit-wars'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

# Bundles built by scripts/pack_for_gpu.sh.
for name in ('code.tgz', 'data.tgz', 'weights.tgz', 'pair_score_assets.tgz'):
    !gsutil cp {BUCKET}/{name} .

# Prior pair-score best.pt that this notebook resumes from. Lands at
# data/runs/pair_score/{PRIOR_RUN}/pair_score_best.pt after extraction.
PRIOR_BEST_LOCAL = f'data/runs/pair_score/{PRIOR_RUN}/pair_score_best.pt'
os.makedirs(os.path.dirname(PRIOR_BEST_LOCAL), exist_ok=True)
!gsutil cp {PRIOR_BEST_GCS} {PRIOR_BEST_LOCAL}

## 4. Unpack + validate

In [ ]:
%cd {WORK}
!tar xzf code.tgz
!tar xzf data.tgz
!tar xzf weights.tgz
!tar xzf pair_score_assets.tgz

import glob
from pathlib import Path

required = (
    ('data/datasets/action', 'action_*.csv', 'action CSVs'),
    ('data/datasets/planet', 'planet_*.csv', 'planet CSVs'),
    ('data/datasets/fleet', 'fleet_*.csv', 'fleet CSVs'),
    ('data/datasets/entity', 'entity_*.csv', 'entity CSVs'),
    ('data/datasets/cross_entity', 'cross_entity_*.csv', 'cross-entity CSVs'),
)
for rel, pattern, label in required:
    count = len(list(Path(rel).glob(pattern)))
    print(f'{label}: {count}')
    if count == 0:
        raise SystemExit(f'no {label} found under {rel}; data.tgz is stale.')

act = sorted(glob.glob('data/runs/action/*/action_best.pt'))
if not act:
    raise SystemExit('no action_*.pt under data/runs/action/ — upload pair_score_assets.tgz.')
ENCODER_CKPT = act[-1]
print('encoder ckpt:', ENCODER_CKPT)

player_replays = sorted(glob.glob(f'data/replays/{PLAYER}/*.json.gz'))
if not player_replays:
    raise SystemExit(f'no replays under data/replays/{PLAYER}/.')
print(f'replays for {PLAYER}: {len(player_replays)}')

if not Path(PRIOR_BEST_LOCAL).exists():
    raise SystemExit(f'prior best.pt missing at {PRIOR_BEST_LOCAL}; section 3 failed.')
print('prior best.pt:', PRIOR_BEST_LOCAL)

## 5. Install + import

In [ ]:
%cd {WORK}
!pip install -q -r requirements.txt --no-deps
!pip install -q kaggle-environments

In [ ]:
import sys
sys.path.insert(0, WORK)

from agents.transformer_v1.pretrain.pair_score import (
    train_pair_score_kwargs,
)
print('train_pair_score_kwargs is the in-kernel entry point — prints stream live.')

### 5b. Materialize the dataset (run once per session)

Parses Ebi's action CSVs into the snapshot tensors `PairScoreStack` consumes and stores them in a kernel variable. **One-time ~3 min cost**; every training re-run below skips the parse and starts immediately.

If you regenerate the action CSVs (e.g. after a featurizer change), restart this cell so the in-memory dataset reflects the new data.

In [ ]:
from agents.transformer_v1.pretrain.pair_score import prepare_dataset

# In-memory dataset. The kernel variable `dataset` will be reused by
# every train cell below — no disk cache needed (saves take longer
# than rebuilding, and the file is multi-GB).
dataset = prepare_dataset(
    player=PLAYER,
    filter_mode='all',
    max_planets=64,
    max_fleets=256,
    n_history=3,
    cache_dir=None,        # in-memory only; flip to a path for crash-recovery
    rebuild_cache=False,
)
print(f'dataset ready: {len(dataset)} snapshots held in kernel')

## 6. Run Stage-2 Frac-only training

Calls `train_pair_score_kwargs(...)` in this Python kernel — every per-epoch line streams into this cell output as it happens. The pair head is frozen, so the **pair val_top1 stays exactly at 0.450** while the frac head learns the launch-fraction distribution from `Ebi`'s acted rows. Expect ~10–20 s/epoch on T4 (only ~33k params train).

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_DIR = f'data/runs/pair_score/exp3_frac_only_{PLAYER}_{TS}'
print('out dir:', OUT_DIR)

# Pass the kernel-resident dataset directly so the trainer skips the
# CSV parse entirely. Iterating on hyperparameters? Re-run this cell
# with new LR / EPOCHS — the data prep step doesn't repeat.
best_ckpt = train_pair_score_kwargs(
    encoder_ckpt=ENCODER_CKPT,
    out_dir=OUT_DIR,
    init_from=PRIOR_BEST_LOCAL,
    frac_weight=FRAC_WEIGHT,
    freeze_pair_head=FREEZE_PAIR_HEAD,
    unfreeze=(UNFREEZE if UNFREEZE else None),
    player=PLAYER,
    filter='all',
    max_rows=MAX_ROWS,
    val_frac=VAL_FRAC,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    device='cuda',
    dataset=dataset,   # in-kernel, skips CSV parse
)
print('done. best ckpt:', best_ckpt)

# Sanity: combined ckpt has pair_score_head + frac_head + all 4 encoders.
import torch
ck = torch.load(best_ckpt, map_location='cpu', weights_only=False)
print('ckpt keys:', sorted(k for k in ck if not k.startswith('_')))

## 7. Log summary — frac metrics as primary, pair as sanity check

Pair metrics should be **identical across all epochs** (head frozen). Frac MAE should drop below the constant-predictor baseline within 5–10 epochs; if it doesn't, the frac head is mis-sized or labels are off.

In [ ]:
import json
log = json.loads(open(f'{WORK}/{OUT_DIR}/log.json').read())
for e in log:
    v = e['val']
    rand = v.get('random_valid_top1', 0.0)
    print(
        f"ep {e['epoch']:2d}  tr_frac_mae={e['train'].get('frac_mae', 0):.3f}  "
        f"val_frac_mae={v.get('frac_mae', 0):.3f}  "
        f"vs_baseline={v.get('frac_baseline_mae', float('nan')):.3f}  "
        f"frac_sigma={v.get('frac_sigma', 0):.3f}  "
        f"||  pair_top1={v['top1']:.3f}  "
        f"dt={e.get('epoch_s', 0):.1f}s"
    )
best = min((e for e in log if 'val' in e),
           key=lambda e: e['val'].get('frac_mae', float('inf')))
print(
    f"\nbest val_frac_mae={best['val'].get('frac_mae', float('nan')):.3f} "
    f"(epoch {best['epoch']}); baseline={best['val'].get('frac_baseline_mae', float('nan')):.3f}"
)

## 8. Push results to GCS

In [ ]:
%cd {WORK}
!gsutil -m cp -r {OUT_DIR} {BUCKET}/runs/
print(f'uploaded {BUCKET}/runs/{OUT_DIR.rsplit("/", 1)[-1]}')